In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
# Change only these four paths
X_TRAIN_PATH = r"C:\Users\tej\OneDrive\Desktop\ML-2.1\data\X_train_processed1.csv"
X_TEST_PATH  = r"C:\Users\tej\OneDrive\Desktop\ML-2.1\data\X_test_processed1.csv"
Y_TRAIN_PATH = r"C:\Users\tej\OneDrive\Desktop\ML-2.1\data\y_train.csv"
Y_TEST_PATH  = r"C:\Users\tej\OneDrive\Desktop\ML-2.1\data\y_test.csv"

feature_cols = ["CGPA", "AptitudeTestScore", "Internships", "Projects", "SoftSkillsRating"]

X_train = pd.read_csv(X_TRAIN_PATH)
X_test  = pd.read_csv(X_TEST_PATH)

y_train = pd.read_csv(Y_TRAIN_PATH)
y_test  = pd.read_csv(Y_TEST_PATH)

X_train = X_train[feature_cols]
X_test  = X_test[feature_cols]

y_train = pd.read_csv(Y_TRAIN_PATH)
y_test  = pd.read_csv(Y_TEST_PATH)

# Convert one-column target DataFrames to Series
if isinstance(y_train, pd.DataFrame) and y_train.shape[1] == 1:
    y_train = y_train.iloc[:, 0]

if isinstance(y_test, pd.DataFrame) and y_test.shape[1] == 1:
    y_test = y_test.iloc[:, 0]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test :", X_test.isna().sum().sum())

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

print("\nPreprocessed feature columns:")
print(list(X_train.columns))


In [ ]:


X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Model-training portion:", X_train.shape)
print("Validation portion   :", X_val.shape)
print("Untouched test set   :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)


In [ ]:


C_values = np.array([0.001, 0.01, 0.1, 1, 10, 100])

L1_RATIO = 0.5

print("C values:", C_values)
print("Elastic Net l1_ratio:", L1_RATIO)


In [ ]:
def compare_regularisation(    X_train, y_train,X_val,y_val, C_values, l1_ratio=0.5, zero_tol=1e-8):
    results = []

    for C in C_values:

        # Lasso: L1
        lasso = LogisticRegression(
            penalty="l1",
            C=C,
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )
        lasso.fit(X_train, y_train)
        lasso_val_acc = accuracy_score(
            y_val, lasso.predict(X_val)
        )
        lasso_nonzero = int(
            np.sum(np.abs(lasso.coef_) > zero_tol)
        )

        # Ridge: L2
        ridge = LogisticRegression(
            penalty="l2",
            C=C,
            solver="lbfgs",
            max_iter=2000,
            random_state=42
        )
        ridge.fit(X_train, y_train)
        ridge_val_acc = accuracy_score(
            y_val, ridge.predict(X_val)
        )
        ridge_nonzero = int(
            np.sum(np.abs(ridge.coef_) > zero_tol)
        )

        # Elastic Net: L1 + L2
        elastic = LogisticRegression(
            penalty="elasticnet",
            C=C,
            l1_ratio=l1_ratio,
            solver="saga",
            max_iter=3000,
            random_state=42
        )
        elastic.fit(X_train, y_train)
        elastic_val_acc = accuracy_score(
            y_val, elastic.predict(X_val)
        )

        results.append({
            "C": C,
            "Lasso_val_acc": lasso_val_acc,
            "Ridge_val_acc": ridge_val_acc,
            "ElasticNet_val_acc": elastic_val_acc,
            "Lasso_nonzero": lasso_nonzero,
            "Ridge_nonzero": ridge_nonzero
        })

    return pd.DataFrame(results)


In [ ]:
results_df = compare_regularisation(
    X_train,
    y_train,
    X_val,
    y_val,
    C_values,
    l1_ratio=L1_RATIO
)

display(results_df)


In [ ]:
plt.figure(figsize=(8, 5))

plt.semilogx(
    results_df["C"],
    results_df["Lasso_val_acc"],
    marker="o",
    label="Lasso"
)

plt.semilogx(
    results_df["C"],
    results_df["Ridge_val_acc"],
    marker="o",
    label="Ridge"
)

plt.semilogx(
    results_df["C"],
    results_df["ElasticNet_val_acc"],
    marker="o",
    label="Elastic Net"
)

plt.xlabel("C (log scale)")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy vs C")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.semilogx(
    results_df["C"],
    results_df["Lasso_nonzero"],
    marker="o",
    label="Lasso"
)

plt.semilogx(
    results_df["C"],
    results_df["Ridge_nonzero"],
    marker="o",
    label="Ridge"
)

plt.xlabel("C (log scale)")
plt.ylabel("Number of Non-Zero Coefficients")
plt.title("Non-Zero Coefficients vs C")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
best_lasso = results_df.loc[results_df["Lasso_val_acc"].idxmax()]
best_ridge = results_df.loc[results_df["Ridge_val_acc"].idxmax()]
best_elastic = results_df.loc[results_df["ElasticNet_val_acc"].idxmax()]

print("Best Lasso:")
display(best_lasso.to_frame().T)

print("Best Ridge:")
display(best_ridge.to_frame().T)

print("Best Elastic Net:")
display(best_elastic.to_frame().T)


In [ ]:
feature_names = list(X_train.columns)

ridge_paths = []
lasso_paths = []
elastic_paths = []

for C in C_values:

    ridge = LogisticRegression(
        penalty="l2",
        C=C,
        solver="lbfgs",
        max_iter=2000,
        random_state=42
    )
    ridge.fit(X_train, y_train)
    ridge_paths.append(ridge.coef_[0])

    lasso = LogisticRegression(
        penalty="l1",
        C=C,
        solver="liblinear",
        max_iter=2000,
        random_state=42
    )
    lasso.fit(X_train, y_train)
    lasso_paths.append(lasso.coef_[0])

    elastic = LogisticRegression(
        penalty="elasticnet",
        C=C,
        l1_ratio=L1_RATIO,
        solver="saga",
        max_iter=3000,
        random_state=42
    )
    elastic.fit(X_train, y_train)
    elastic_paths.append(elastic.coef_[0])

ridge_paths = np.array(ridge_paths)
lasso_paths = np.array(lasso_paths)
elastic_paths = np.array(elastic_paths)

print("Number of model features:", len(feature_names))
print("Ridge path shape:", ridge_paths.shape)
print("Lasso path shape:", lasso_paths.shape)
print("Elastic Net path shape:", elastic_paths.shape)


In [ ]:
plt.figure(figsize=(10, 6))

for j, feature in enumerate(feature_names):
    plt.semilogx(
        C_values,
        ridge_paths[:, j],
        marker="o",
        label=feature
    )

plt.axhline(0, linestyle="--")
plt.xlabel("C (log scale)")
plt.ylabel("Coefficient")
plt.title("Ridge Logistic Regression Shrinkage Paths")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

for j, feature in enumerate(feature_names):
    plt.semilogx(
        C_values,
        lasso_paths[:, j],
        marker="o",
        label=feature
    )

plt.axhline(0, linestyle="--")
plt.xlabel("C (log scale)")
plt.ylabel("Coefficient")
plt.title("Lasso Logistic Regression Shrinkage Paths")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

for j, feature in enumerate(feature_names):
    plt.semilogx(
        C_values,
        elastic_paths[:, j],
        marker="o",
        label=feature
    )

plt.axhline(0, linestyle="--")
plt.xlabel("C (log scale)")
plt.ylabel("Coefficient")
plt.title(f"Elastic Net Shrinkage Paths (l1_ratio={L1_RATIO})")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
best_C_lasso = float(best_lasso["C"])
best_C_ridge = float(best_ridge["C"])
best_C_elastic = float(best_elastic["C"])

lasso_best = LogisticRegression(
    penalty="l1",
    C=best_C_lasso,
    solver="liblinear",
    max_iter=2000,
    random_state=42
)
lasso_best.fit(X_train, y_train)

ridge_best = LogisticRegression(
    penalty="l2",
    C=best_C_ridge,
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)
ridge_best.fit(X_train, y_train)

elastic_best = LogisticRegression(
    penalty="elasticnet",
    C=best_C_elastic,
    l1_ratio=L1_RATIO,
    solver="saga",
    max_iter=3000,
    random_state=42
)
elastic_best.fit(X_train, y_train)

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Ridge": ridge_best.coef_[0],
    "Lasso": lasso_best.coef_[0],
    "ElasticNet": elastic_best.coef_[0]
})

display(coefficient_table)


In [ ]:
best_candidates = {
    "Lasso": (best_C_lasso, float(best_lasso["Lasso_val_acc"])),
    "Ridge": (best_C_ridge, float(best_ridge["Ridge_val_acc"])),
    "ElasticNet": (best_C_elastic, float(best_elastic["ElasticNet_val_acc"]))
}

best_method = max(
    best_candidates,
    key=lambda name: best_candidates[name][1]
)

if best_method == "Lasso":
    final_model = lasso_best
elif best_method == "Ridge":
    final_model = ridge_best
else:
    final_model = elastic_best

y_test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Selected model:", best_method)
print("Selected C:", best_candidates[best_method][0])
print("Validation accuracy:", best_candidates[best_method][1])
print("Final test accuracy:", test_accuracy)
